In [ ]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community langchain-text-splitters networkx pypdf faiss-cpu fastapi uvicorn
!pip install langchain==0.1.17
!pip install langchain-openai==0.0.6
!pip install langchain-core==0.1.53 
!pip install langchain-community==0.0.33
!pip install networkx
!pip install pypdf
!pip install faiss-cpu
!pip install fastapi uvicorn

In [ ]:
# If that still fails, ensure langchain is actually installed in your environment:
import langchain; print(langchain.__version__)
import langchain_core; print(langchain_core.__version__)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader

In [ ]:
# 2️⃣ Load PDF
loader = PyPDFLoader("HR_Policy_Manual_2023.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

In [ ]:
# 3️⃣ Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

In [ ]:
# 4️⃣ Create embeddings + FAISS index
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# 5️⃣ Build RetrievalQA chain


# 5. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 6. Build RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # simple chain type
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

In [ ]:
# 6️⃣ Ask a question
query = "Summarize the key points from the HR policy manual"
result = qa.invoke(query)

print("Answer:", result["result"])
print("Sources:", result["source_documents"])